# Step 1: Install Required Libraries in Kaggle

In [1]:
!pip install -q opencv-python torchaudio pandas

# Step 2: Load FakeAVCelebDataset

In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import librosa
import torchaudio.transforms as AT
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==============================================================================
# 1. HELPER TRANSFORMS & AUGMENTATIONS
# ==============================================================================
class SpecAugment(torch.nn.Module):
    """SpecAugment for Mel-Spectrogram Feature Masking"""
    def __init__(self, freq_mask_param=15, time_mask_param=35):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def forward(self, x):
        if not self.training:
            return x
        # x shape: (1, seq_len, feature_dim)
        f_len = x.shape[2]
        f = np.random.randint(0, self.freq_mask_param)
        f0 = np.random.randint(0, max(1, f_len - f))
        x[:, :, f0:f0+f] = 0.0

        t_len = x.shape[1]
        t = np.random.randint(0, self.time_mask_param)
        t0 = np.random.randint(0, max(1, t_len - t))
        x[:, t0:t0+t, :] = 0.0
        return x

def get_visual_transforms(is_train=True):
    """Standard ImageNet preprocessing pipeline for video frames."""
    if is_train:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# ==============================================================================
# 2. PRODUCTION FAKEAVCELEB DATASET CLASS
# ==============================================================================
class FakeAVCelebDataset(Dataset):
    """
    Production PyTorch Dataset for FakeAVCeleb (Phase 2 TSA-Net Pipeline)
    """
    def __init__(self, metadata_csv, dataset_root, seq_len=16, is_train=True):
        self.dataset_root = dataset_root
        self.seq_len = seq_len
        self.is_train = is_train

        # Load metadata CSV
        self.df = pd.read_csv(metadata_csv)
        
        # PyTorch Mel-Spectrogram Generator
        self.mel_transform = AT.MelSpectrogram(
            sample_rate=16000,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )
        
        self.v_transform = get_visual_transforms(is_train)
        self.spec_aug = SpecAugment() if is_train else torch.nn.Identity()

    def __len__(self):
        return len(self.df)

    def _sample_video_frames(self, video_path):
        """Uniformly samples seq_len frames using OpenCV frame seek."""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Fallback tensor if video stream is unreadable
        if not cap.isOpened() or total_frames <= 0:
            cap.release()
            return torch.zeros((self.seq_len, 3, 224, 224))

        frame_indices = np.linspace(0, total_frames - 1, self.seq_len, dtype=int)
        frames = []

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            success, frame = cap.read()
            if success and frame is not None:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                tensor_frame = self.v_transform(frame_rgb)
                frames.append(tensor_frame)
            else:
                frames.append(torch.zeros((3, 224, 224)))

        cap.release()
        return torch.stack(frames, dim=0)

    def _extract_audio_features(self, video_path):
        """Extracts audio safely from .mp4 container using librosa."""
        try:
            y, sr = librosa.load(video_path, sr=16000, mono=True)
            waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)

            # Extract Mel-Spectrogram
            mel_spec = self.mel_transform(waveform).squeeze(0)  # Shape: (128, time_steps)
            mel_spec = mel_spec.transpose(0, 1)                  # Shape: (time_steps, 128)

            # Interpolate time dimension to match seq_len (T=16)
            mel_spec = mel_spec.unsqueeze(0).unsqueeze(0)        # Shape: (1, 1, time_steps, 128)
            mel_spec = torch.nn.functional.interpolate(
                mel_spec, size=(self.seq_len, 128), mode='bilinear', align_corners=False
            ).squeeze(0).squeeze(0)                             # Shape: (seq_len, 128)

            return mel_spec
        except Exception:
            # Fallback tensor if audio reading fails or file is silent
            return torch.zeros((self.seq_len, 128))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Build path relative to dataset root
        video_rel_path = str(row['path']) if 'path' in row else str(row['filename'])
        full_video_path = os.path.join(self.dataset_root, video_rel_path)

        # Label Parsing: RealVideo-RealAudio -> 0.0 (Real), All others -> 1.0 (Fake)
        category = str(row['type']) if 'type' in row else str(row['category'])
        label = 0.0 if "RealVideo-RealAudio" in category else 1.0

        # Load Features
        video_tensor = self._sample_video_frames(full_video_path)
        audio_tensor = self._extract_audio_features(full_video_path)
        
        # SpecAugment applied during training
        audio_tensor = self.spec_aug(audio_tensor.unsqueeze(0)).squeeze(0)

        return video_tensor, audio_tensor, torch.tensor(label, dtype=torch.float32)

# ==============================================================================
# 3. DYNAMIC PATH DISCOVERY & DATALOADER CREATION
# ==============================================================================
def find_fakeavceleb_paths():
    """Automatically locates meta_data.csv inside Kaggle input directory."""
    search_root = "/kaggle/input"
    if not os.path.exists(search_root):
        search_root = "."

    csv_path, dataset_root = None, None

    for root, _, files in os.walk(search_root):
        if "meta_data.csv" in files:
            csv_path = os.path.join(root, "meta_data.csv")
            dataset_root = root
            break

    if csv_path is None:
        raise FileNotFoundError(
            "Could not locate 'meta_data.csv'. Please check '+ Add Data' on Kaggle."
        )

    print(f"✓ Located meta_data.csv at: {csv_path}")
    print(f"✓ Dataset Root folder set to: {dataset_root}")
    return csv_path, dataset_root

# Automatically search for paths
METADATA_CSV_PATH, KAGGLE_DATASET_ROOT = find_fakeavceleb_paths()

# Instantiate Phase 2 Datasets
train_dataset = FakeAVCelebDataset(
    metadata_csv=METADATA_CSV_PATH, 
    dataset_root=KAGGLE_DATASET_ROOT, 
    seq_len=16, 
    is_train=True
)

val_dataset = FakeAVCelebDataset(
    metadata_csv=METADATA_CSV_PATH, 
    dataset_root=KAGGLE_DATASET_ROOT, 
    seq_len=16, 
    is_train=False
)

# Instantiate PyTorch DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

✓ Located meta_data.csv at: /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/meta_data.csv
✓ Dataset Root folder set to: /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2


# Step 3: Instantiating in Kaggle Notebook

In [3]:
from torch.utils.data import random_split

# 1. Instantiate full dataset once
full_dataset = FakeAVCelebDataset(
    metadata_csv=METADATA_CSV_PATH, 
    dataset_root=KAGGLE_DATASET_ROOT, 
    seq_len=16, 
    is_train=True
)

# 2. 80/20 Train/Val Split
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) # Fixed seed for reproducibility
)

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"✓ Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

✓ Train Batches: 2157 | Val Batches: 540
